# 12: Train lesion segmentation and measure site bias

Trains a U-Net (ResNet-34 encoder) that outlines four lesion types (microaneurysms,
haemorrhages, hard exudates, soft exudates) on IDRiD (India, one camera) and DDR (China, many
cameras), three times:

| Run | Trained on | Selected on | Question |
|---|---|---|---|
| `seg-pooled` | IDRiD + DDR train | IDRiD + DDR val | the production model |
| `seg-ddr_only` | DDR train | DDR val | how well does it transfer to IDRiD? |
| `seg-idrid_only` | IDRiD train | IDRiD val | how well does it transfer to DDR? |

Then every model segments every image it did not train on, and the metrics (Dice and AUPR per
lesion, per site, with bootstrap 95% CIs; one-site models against the pooled one, paired) are
computed. Runs already finished in an attached earlier output are skipped; an unfinished one
resumes.

**Settings** (right-hand panel): Accelerator **GPU T4 x1**, Internet **On**,
Persistence **Files only**.
**Inputs** (*Add Input*):
- *Datasets* → *Your Datasets* → **retinaguard-idrid-seg** (prepared IDRiD lesion images)
- To **resume**: also the output of this notebook's previous version.

DDR's lesion set (757 images, CC BY-NC-SA 4.0) is downloaded from the Hugging Face copy of the
authors' release and prepared here.

**Output** (`/kaggle/working/`): `seg/ddr/` (prepared DDR set), `runs/seg-*/` (`best.pt`,
`last.pt`, `metrics.jsonl`, `stats.npz`, ...), `seg_eval/` (`metrics.csv`, `sites.csv`,
`comparison.csv`).

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
DDR_REPO = "addone5/DDR-dataset"  # Hugging Face copy of the DDR release
RUNS = {
    "seg-pooled": ["idrid", "ddr"],
    "seg-ddr_only": ["ddr"],
    "seg-idrid_only": ["idrid"],
}
OVERRIDES = []  # Hydra overrides for every run, e.g. ["train.epochs=30"]

In [ ]:
EXTRAS = '"pandera>=0.20" "hydra-core>=1.3" "timm>=1.0" "albumentations>=2.0"'
EXTRAS += ' "segmentation-models-pytorch>=0.4"'
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" {EXTRAS}

In [ ]:
# --- 1. Inputs: IDRiD (attached) and DDR (downloaded and prepared once) -----------
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

from retinaguard import __version__
from retinaguard.data.segmentation import INDEX_NAME

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
RUNS_DIR = WORKING / "runs"
idrid = sorted(p.parent for p in INPUT_ROOT.glob(f"**/{INDEX_NAME}") if "idrid" in str(p))
assert idrid, "IDRiD lesion set not found: attach the retinaguard-idrid-seg dataset"

ddr_dir = WORKING / "seg" / "ddr"
earlier = sorted(p.parent for p in INPUT_ROOT.glob(f"**/seg/ddr/{INDEX_NAME}"))
if earlier and not (ddr_dir / INDEX_NAME).exists():
    shutil.copytree(earlier[0], ddr_dir)
if not (ddr_dir / INDEX_NAME).exists():
    raw = Path(
        snapshot_download(
            DDR_REPO,
            repo_type="dataset",
            allow_patterns=["lesion_segmentation/*"],
            local_dir="/tmp/ddr",
        )
    )
    # in a separate process, so this one never runs OpenCV's threads before training forks
    subprocess.run(
        [
            sys.executable,
            "-m",
            "retinaguard.data.segmentation",
            "ddr",
            str(raw / "lesion_segmentation"),
            str(ddr_dir),
        ],
        check=True,
    )
    shutil.rmtree("/tmp/ddr")
for run in RUNS:
    previous = sorted(INPUT_ROOT.glob(f"**/runs/{run}/last.pt"))
    if previous and not (RUNS_DIR / run / "last.pt").exists():
        shutil.copytree(previous[0].parent, RUNS_DIR / run, dirs_exist_ok=True)
        print("resuming", run, "from", previous[0].parent)

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("\t")[0]
assert torch.cuda.is_available(), (
    f"no GPU (torch {torch.__version__}): set Accelerator to GPU in the notebook settings, "
    "then save a new version"
)
print(f"retinaguard {__version__} @ {git_sha[:10]}, GPU {torch.cuda.get_device_name(0)}")
SHARD_DIRS = [*idrid, ddr_dir]
print("lesion sets:", [str(d) for d in SHARD_DIRS])

In [ ]:
# --- 2. Train and segment, one run after another --------------------------------
from retinaguard.config import load_config
from retinaguard.tracking import RunLogger
from retinaguard.train.train_seg import predict_run, train

for run, sites in RUNS.items():
    if (RUNS_DIR / run / "stats.npz").exists():
        print(run, "already finished")
        continue
    cfg = load_config(
        "train/segmentation",
        [
            f"run_name={run}",
            f"output_dir={RUNS_DIR}",
            f"data.train_sites=[{','.join(sites)}]",
            f"data.shard_dirs=[{','.join(str(d) for d in SHARD_DIRS)}]",
            *OVERRIDES,
        ],
    )
    RunLogger(RUNS_DIR / run).set_tags(
        {"git_sha": git_sha, "gpu": torch.cuda.get_device_name(0), "platform": "kaggle"}
    )
    run_dir = train(cfg)
    rows = predict_run(cfg, run_dir)
    print(f"{run}: {len(rows)} images segmented")

In [ ]:
# --- 3. Learning curves --------------------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(1, len(RUNS), figsize=(14, 3.5), sharey=True)
for ax, run in zip(axes, RUNS, strict=True):
    metrics = pd.DataFrame(RunLogger(RUNS_DIR / run).read_metrics())
    table = metrics.pivot_table(index="step", columns="key", values="value", aggfunc="last")
    best = torch.load(RUNS_DIR / run / "best.pt", map_location="cpu", weights_only=False)
    print(f"{run}: best epoch {best['epoch']}, val score {best['val_score']:.4f}")
    shown = [c for c in table.columns if c.startswith("val_aupr_")]
    table[shown].dropna().plot(ax=ax, title=run, legend=ax is axes[0])
    ax.set_xlabel("epoch")
axes[0].set_ylabel("validation AUPR")
plt.tight_layout()
plt.show()

In [ ]:
# --- 4. Metrics, site gaps and paired comparisons ---------------------------------
from retinaguard.train.evaluate_seg import main as evaluate_seg

evaluate_seg(
    [
        *(str(RUNS_DIR / run) for run in RUNS),
        "--reference=seg-pooled",
        f"--out-dir={WORKING / 'seg_eval'}",
    ]
)
print(pd.read_csv(WORKING / "seg_eval" / "sites.csv").round(3).to_string(index=False))

**After the version finishes:** keep the version; its output holds the runs and the metrics.
Do not download the `.pt` files into the repository.